# Walidacja danych po preprocessingu

Sprawdzenie struktury i spójności wyjść pipeline'u dla dwóch wariantów:

- `subsample_1_32_clicks_only` — GRU4Rec, TAGNN, TGN (tylko kliknięcia)
- `subsample_1_32_with_buys` — TGN z kliknięciami i zakupami

Oczekiwany katalog bazowy: `data/processed/`.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.common.paths import get_project_root

ROOT = get_project_root()
PROCESSED = ROOT / 'data' / 'processed'

CLICKS_ONLY = PROCESSED / 'subsample_1_32_clicks_only'
WITH_BUYS = PROCESSED / 'subsample_1_32_with_buys'

SPLITS = ['train', 'val', 'test_internal']

EXPECTED_CLICKS_ONLY = {
    'train': ['gru4rec_examples.parquet', 'tagnn_examples.pkl', 'tgn/events.parquet', 'tgn/examples.parquet'],
    'val': ['gru4rec_examples.parquet', 'tagnn_examples.pkl', 'tgn/events.parquet', 'tgn/examples.parquet'],
    'test_internal': ['gru4rec_examples.parquet', 'tagnn_examples.pkl', 'tgn/events.parquet', 'tgn/examples.parquet'],
    'challenge_test': ['gru4rec_examples.parquet', 'tagnn_examples.parquet', 'tgn/events.parquet', 'tgn/examples.parquet'],
}

EXPECTED_WITH_BUYS = {
    'train': ['tgn/events.parquet', 'tgn/examples.parquet'],
    'val': ['tgn/events.parquet', 'tgn/examples.parquet'],
    'test_internal': ['tgn/events.parquet', 'tgn/examples.parquet'],
    'challenge_test': ['tgn/events.parquet', 'tgn/examples.parquet'],
}

TGN_EVENT_COLS = [
    'event_id', 'session_id', 'session_idx', 'item_id', 'item_idx_tgn',
    't_sec', 'event_type', 'cat_bucket_idx', 'price_log', 'quantity', 'is_click', 'split',
]

GRU_COLS = ['example_id', 'split', 'session_id', 'target_click_pos', 'history_item_idx', 'target_item_idx']
TGN_EX_COLS = [
    'example_id', 'split', 'session_id', 'session_idx', 'target_item_idx_tgn',
    'target_t_sec', 'target_event_id', 'prefix_last_event_id', 'prefix_num_events',
]


def load_meta(path: Path) -> dict:
    return json.loads((path / 'meta.json').read_text(encoding='utf-8'))


def t_sec_monotonic_within_sessions(df: pd.DataFrame) -> bool:
    """t_sec musi rosnąć w obrębie sesji (globalnie może maleć — sesje się nakładają w czasie)."""
    return bool(
        df.groupby('session_id', sort=False)['t_sec']
        .apply(lambda s: s.is_monotonic_increasing)
        .all()
    )

## 1. Obecność katalogów i plików

In [ ]:
def check_tree(base: Path, expected: dict[str, list[str]]) -> pd.DataFrame:
    rows = []
    for split, files in expected.items():
        for rel in ['meta.json', 'vocab/item_vocab.json', 'vocab/cat_bucket2idx.json']:
            p = base / rel
            rows.append({'split': split, 'file': rel, 'exists': p.exists(), 'bytes': p.stat().st_size if p.exists() else 0})
        for rel in files:
            p = base / split / rel
            rows.append({'split': split, 'file': rel, 'exists': p.exists(), 'bytes': p.stat().st_size if p.exists() else 0})
    return pd.DataFrame(rows)

for name, base, expected in [
    ('clicks_only', CLICKS_ONLY, EXPECTED_CLICKS_ONLY),
    ('with_buys', WITH_BUYS, EXPECTED_WITH_BUYS),
]:
    print(f'\n=== {name} ===')
    df_tree = check_tree(base, expected)
    display(df_tree)
    missing = df_tree[~df_tree['exists']]
    if len(missing):
        print('BRAKUJĄCE:', missing['file'].tolist())
    else:
        print('OK: wszystkie oczekiwane pliki istnieją')

## 2. Porównanie meta.json (oba warianty)

In [ ]:
meta_clicks = load_meta(CLICKS_ONLY)
meta_buys = load_meta(WITH_BUYS)

compare_keys = [
    ('subsample_fraction', meta_clicks['config']['subsample_fraction'], meta_buys['config']['subsample_fraction']),
    ('n_sessions', meta_clicks['stats']['n_sessions'], meta_buys['stats']['n_sessions']),
    ('n_clicks', meta_clicks['stats']['n_clicks'], meta_buys['stats']['n_clicks']),
    ('n_items_vocab', meta_clicks['stats']['n_items_train_vocab'], meta_buys['stats']['n_items_train_vocab']),
    ('train_end', meta_clicks['boundaries']['train_end'], meta_buys['boundaries']['train_end']),
    ('train_tgn_examples', meta_clicks['stats']['train_examples']['tgn'], meta_buys['stats']['train_examples']['tgn']),
    ('train_events_clicks', meta_clicks['stats']['train_examples']['events'], meta_buys['stats']['train_examples']['events']),
]

df_meta = pd.DataFrame(compare_keys, columns=['metric', 'clicks_only', 'with_buys'])
df_meta['match'] = df_meta['clicks_only'] == df_meta['with_buys']
display(df_meta)

print('include_buys:', meta_clicks['config']['include_buys'], 'vs', meta_buys['config']['include_buys'])
print('sequence_models export (clicks_only):', meta_clicks['exports']['sequence_models'])
print('tgn_events_include_buys:', meta_buys['exports']['tgn_events_include_buys'])

extra_buys_train = meta_buys['stats']['train_examples']['events'] - meta_clicks['stats']['train_examples']['events']
print(f'\nDodatkowe zdarzenia w train (with_buys vs clicks_only): {extra_buys_train:,}')

## 3. Vocab — identyczność między wariantami

In [ ]:
v_clicks = json.loads((CLICKS_ONLY / 'vocab/item_vocab.json').read_text(encoding='utf-8'))
v_buys = json.loads((WITH_BUYS / 'vocab/item_vocab.json').read_text(encoding='utf-8'))

print('n_items:', v_clicks['n_items'], v_buys['n_items'])
print('item2idx identical:', v_clicks['item2idx'] == v_buys['item2idx'])
print('GRU unk_idx:', v_clicks['unk_gru_idx'], '| TGN unk_idx:', v_clicks['unk_tgn_idx'])

## 4. Schematy kolumn (train)

In [ ]:
def assert_cols(df: pd.DataFrame, expected: list[str], label: str) -> None:
    missing = set(expected) - set(df.columns)
    extra = set(df.columns) - set(expected)
    print(f'{label}: rows={len(df):,}')
    if missing:
        print('  MISSING cols:', sorted(missing))
    if extra:
        print('  EXTRA cols:', sorted(extra))
    if not missing:
        print('  OK: wszystkie wymagane kolumny')

# clicks_only — train
train_gru = pd.read_parquet(CLICKS_ONLY / 'train/gru4rec_examples.parquet')
train_tgn_ev = pd.read_parquet(CLICKS_ONLY / 'train/tgn/events.parquet')
train_tgn_ex = pd.read_parquet(CLICKS_ONLY / 'train/tgn/examples.parquet')

with (CLICKS_ONLY / 'train/tagnn_examples.pkl').open('rb') as fh:
    train_tagnn = pickle.load(fh)

train_tagnn_df = pd.DataFrame(train_tagnn)

assert_cols(train_gru, GRU_COLS, 'GRU4Rec train')
assert_cols(train_tgn_ev, TGN_EVENT_COLS, 'TGN events (clicks_only)')
assert_cols(train_tgn_ex, TGN_EX_COLS, 'TGN examples (clicks_only)')
print('TAGNN train records:', len(train_tagnn_df))
print('TAGNN columns:', train_tagnn_df.columns.tolist())

## 5. Konwencje indeksów i przykładowe rekordy

In [ ]:
n_items = v_clicks['n_items']
pad_idx = v_clicks['pad_idx']
unk_gru = v_clicks['unk_gru_idx']
unk_tgn = v_clicks['unk_tgn_idx']

sample = train_gru.iloc[0]
print('GRU sample:')
print('  history_item_idx:', sample['history_item_idx'])
print('  target_item_idx:', sample['target_item_idx'])

gru_targets = train_gru['target_item_idx']
gru_in_range = ((gru_targets >= 1) & (gru_targets <= n_items)) | (gru_targets == unk_gru)
print(f'GRU target idx valid: {gru_in_range.mean()*100:.2f}%')

tgn_items = train_tgn_ev['item_idx_tgn']
tgn_in_range = ((tgn_items >= 0) & (tgn_items < n_items)) | (tgn_items == unk_tgn)
print(f'TGN item_idx valid: {tgn_in_range.mean()*100:.2f}%')

# clicks_only: same event types
print('event_type unique (clicks_only):', sorted(train_tgn_ev['event_type'].unique()))
print('is_click share:', train_tgn_ev['is_click'].mean())

# TAGNN: item_ids length = history_len + 1
tagnn_lens_ok = (train_tagnn_df['history_len'] + 1 == train_tagnn_df['item_ids'].map(len)).mean()
print(f'TAGNN item_ids length OK: {tagnn_lens_ok*100:.2f}%')

## 6. TGN: clicks_only vs with_buys (train)

In [ ]:
train_ev_buys = pd.read_parquet(WITH_BUYS / 'train/tgn/events.parquet')

rows = []
for label, df in [('clicks_only', train_tgn_ev), ('with_buys', train_ev_buys)]:
    rows.append({
        'variant': label,
        'n_events': len(df),
        'n_clicks': int(df['is_click'].sum()),
        'n_buys': int((~df['is_click']).sum()),
        'n_sessions': df['session_idx'].nunique(),
        't_sec ok w sesji': t_sec_monotonic_within_sessions(df),
        't_sec monotonic (global)': bool(df['t_sec'].is_monotonic_increasing),
        'event_id unique': df['event_id'].is_unique,
    })

display(pd.DataFrame(rows))
print(
    'Uwaga: globalne t_sec NIE musi rosnąć — plik jest posortowany po session_id, '
    'a sesje nakładają się w czasie. TGN przy treningu sortuje po t_sec.'
)

if len(train_ev_buys) <= len(train_tgn_ev):
    print('UWAGA: with_buys powinien mieć więcej zdarzeń niż clicks_only')
else:
    print(f'OK: +{len(train_ev_buys) - len(train_tgn_ev):,} zdarzeń (zakupy)')

print('\nRozkład event_type (with_buys):')
display(train_ev_buys['event_type'].value_counts())

## 7. Liczności przykładów vs meta.json

In [ ]:
def check_counts(base: Path, meta: dict) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        n_gru = len(pd.read_parquet(base / split / 'gru4rec_examples.parquet'))
        n_tgn_ex = len(pd.read_parquet(base / split / 'tgn/examples.parquet'))
        n_ev = len(pd.read_parquet(base / split / 'tgn/events.parquet'))
        key = f'{split}_examples'
        exp = meta['stats'][key]
        rows.append({
            'split': split,
            'gru_file': n_gru,
            'gru_meta': exp['gru4rec'],
            'tgn_ex_file': n_tgn_ex,
            'tgn_ex_meta': exp['tgn'],
            'events_file': n_ev,
            'events_meta': exp['events'],
            'match': n_gru == exp['gru4rec'] == n_tgn_ex == exp['tgn'],
        })
    return pd.DataFrame(rows)

print('clicks_only:')
display(check_counts(CLICKS_ONLY, meta_clicks))

# train >> val — sliding window vs last-click
train_n = meta_clicks['stats']['train_examples']['tgn']
val_n = meta_clicks['stats']['val_examples']['tgn']
print(f'\nStosunek train/val examples: {train_n / val_n:.1f}x (oczekiwane: dużo więcej train przez sliding window)')

## 8. Challenge test (bez subsample)

In [ ]:
ch_events = pd.read_parquet(CLICKS_ONLY / 'challenge_test/tgn/events.parquet')
ch_examples = pd.read_parquet(CLICKS_ONLY / 'challenge_test/tgn/examples.parquet')

print('challenge events:', f'{len(ch_events):,}')
print('challenge examples:', f'{len(ch_examples):,}')
print('meta events:', f"{meta_clicks['stats']['challenge_test_examples']['events']:,}")
print('meta examples:', f"{meta_clicks['stats']['challenge_test_examples']['tgn']:,}")
print('cold start items (meta):', meta_clicks['stats']['challenge_test_examples']['cold_start_items'])

# 1 example per session (last click)
n_sessions_ch = ch_events['session_id'].nunique()
ratio = len(ch_examples) / n_sessions_ch
print(f'\nPrzykłady / sesje: {ratio:.3f} (≈1.0 jeśli większość sesji ma >=2 kliknięcia)')

# TGN prefix spójność (losowa próbka)
sample_ex = ch_examples.sample(5, random_state=42)
for _, row in sample_ex.iterrows():
    prefix = ch_events[ch_events['event_id'] <= row['prefix_last_event_id']]
    ok = len(prefix) == row['prefix_num_events']
    print(f"session {row['session_id']}: prefix_num={row['prefix_num_events']}, ok={ok}")

## 9. Podsumowanie checklisty

In [ ]:
checks = []


def add(ok: bool, ok_msg: str, fail_msg: str):
    checks.append({'check': ok_msg if ok else fail_msg, 'pass': bool(ok)})


add(CLICKS_ONLY.exists() and WITH_BUYS.exists(), 'Oba katalogi processed istnieją', 'Brak katalogu processed')
add(v_clicks['item2idx'] == v_buys['item2idx'], 'Vocab identyczny między wariantami', 'Vocab różny!')
add(meta_clicks['config']['include_buys'] is False, 'clicks_only: include_buys=False', 'Zła flaga clicks_only')
add(meta_buys['config']['include_buys'] is True, 'with_buys: include_buys=True', 'Zła flaga with_buys')
add(len(train_ev_buys) > len(train_tgn_ev), 'with_buys ma więcej zdarzeń train', 'Brak dodatkowych buys w train')
add(train_tgn_ev['is_click'].all(), 'clicks_only: wszystkie zdarzenia to kliknięcia', 'Nieoczekiwane buys w clicks_only')
add(train_tgn_ev['event_id'].is_unique, 'event_id unikalne (train clicks_only)', 'Duplikaty event_id')
add(
    t_sec_monotonic_within_sessions(train_tgn_ev),
    't_sec rosnące w obrębie każdej sesji (train)',
    't_sec niespójne w sesji',
)

df_checks = pd.DataFrame(checks)
display(df_checks)

n_pass = df_checks['pass'].sum()
print(f'\nWynik: {n_pass}/{len(df_checks)} testów zaliczonych')